# Tech Challenge 3 - Big Data & Analytics
## State of Data Brasil (2023, 2024 e 2025)

Este notebook analisa as três edições mais recentes da pesquisa State of Data Brasil, realizada por Data Hackers e Bain.

O objetivo é reunir os dados dos três anos e responder às seguintes perguntas:

1. Como está estruturado o mercado brasileiro de Dados?
2. Quais perfis profissionais são mais valorizados pelo mercado?
3. Como está a diversidade de gênero nas carreiras de Dados?
4. Quais tecnologias apresentam maior adoção entre os profissionais?
5. Qual é o nível de adoção de Inteligência Artificial e qual é o seu impacto?
6. Existem diferenças entre regiões, níveis de senioridade ou modelos de trabalho?
7. Quais oportunidades e desafios existem para empresas que desejam investir em Dados e Inteligência Artificial?

**Origem dos dados.** O pipeline em AWS (Glue + Athena, descrito no `README.md` da `fase-3`) processa os dados brutos da pesquisa pelas camadas Raw → Cleansed → Transformed → Curated.

Para responder às 7 perguntas do desafio com aprofundamento analítico e cruzamentos multidimensionais (como *Cargo × Senioridade × Gênero × Faixa Salarial*), as consultas analíticas SQL foram executadas via **Amazon Athena diretamente sobre a camada Transformed** (que preserva a granularidade atômica, respondente a respondente). Os resultados gerados pelas queries foram exportados e organizados por pergunta nas pastas `data/outputs/pergunta-1/` a `data/outputs/pergunta-7/`.

Este notebook carrega essas tabelas analíticas para gerar as análises e os gráficos a seguir.

In [ ]:
# Importando bibliotecas necessárias.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import textwrap

# Configurações gerais
pd.set_option("display.max_columns", 50)

# Paleta de cores usada nos gráficos.
COR_DESTAQUE = "#5C2F7C"    # Cor usada para destacar um dado importante.
COR_COMPARACAO = "#A5114EE5"  # Cor principal para a série mais recente.
COR_SEC_1 = "#667985"
COR_SEC_2 = "#8D9AA1"
COR_SEC_3 = "#B3BBBF"

COR_SEQUENCIAL = COR_COMPARACAO
PALETA_CATEGORICA = [COR_COMPARACAO, COR_SEC_1, COR_SEC_2, COR_SEC_3, COR_DESTAQUE]
TINTA_PRIMARIA = "#161616"
TINTA_SECUNDARIA = "#4A4A47"
TINTA_MUTED = "#6F6E69"
GRADE = "#E1E0D9"
SUPERFICIE = "#FCFCFB"

# Configurações de estilo do matplotlib.
plt.rcParams.update({
    "figure.facecolor": SUPERFICIE,
    "axes.facecolor": SUPERFICIE,
    "axes.edgecolor": GRADE,
    "axes.labelcolor": TINTA_SECUNDARIA,
    "axes.titlecolor": TINTA_PRIMARIA,
    "text.color": TINTA_PRIMARIA,
    "xtick.color": TINTA_MUTED,
    "ytick.color": TINTA_MUTED,
    "grid.color": GRADE,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "figure.dpi": 110,
})

ANOS = [2023, 2024, 2025]

# Usa cinza nos anos antigos e a cor principal no ano mais recente.
COR_POR_ANO = {2023: COR_SEC_3, 2024: COR_SEC_2, 2025: COR_COMPARACAO}

## 1. Carregamento das tabelas analíticas (consultas Athena sobre a camada Transformed)

Cada pergunta do desafio tem sua própria pasta em `data/outputs/`, contendo as tabelas geradas a partir das consultas SQL no Amazon Athena (`src/sql/queries_athena.sql`) executadas sobre a tabela harmonizada `db_state_of_data.yearly` na camada **Transformed**.

**Por que realizamos as consultas analíticas na camada Transformed e não na Curated?**
- **Granularidade atômica e cruzamentos multidimensionais:** A camada *Transformed* preserva cada registro individual da pesquisa (1 linha = 1 respondente) com todos os tipos harmonizados e variáveis canônicas unificadas. Isso permite cruzar múltiplas dimensões em simultâneo conforme exigido pelas perguntas do desafio (por exemplo, avaliar *Cargo × Senioridade × Gênero × Faixa Salarial* na Pergunta 3, ou *Tecnologia × Senioridade* na Pergunta 6).
- **Limitação de escopo da camada Curated pré-agregada:** A camada *Curated* é composta por datamarts com agregações pré-calculadas e isoladas por dimensão (ex.: tabela exclusiva de gênero, tabela exclusiva de setor). Consultar apenas tabelas pré-agregadas impediria análises cruzadas ad-hoc complexas entre variáveis.
- **Eficiência e reprodutibilidade:** O Athena atuou como motor de agregação sobre a base canônica atômica (*Transformed*), gerando os recortes exatos e os percentuais necessários para suportar os gráficos deste notebook de forma leve e direta.

Essas tabelas chegam ao notebook já agregadas e estruturadas para a etapa visual, exigindo apenas os pequenos ajustes residuais documentados na seção seguinte.

In [ ]:
# Caminho para as tabelas analíticas exportadas do Athena, organizadas por pergunta.
BASE_OUTPUTS = "../data/outputs"

def carregar(pergunta, arquivo):
    # Carrega uma tabela analítica de uma das pastas pergunta-N.
    caminho = f"{BASE_OUTPUTS}/pergunta-{pergunta}/{arquivo}"
    return pd.read_csv(caminho, encoding="utf-8")

# Pergunta 1 - estrutura do mercado
p1_senioridade = carregar(1, "p1_senioridade.csv")
p1_setor = carregar(1, "p1_setor.csv")
p1_tamanho_time = carregar(1, "p1_tamanho-time.csv")

# Pergunta 2 - perfis valorizados (salário)
p2_faixa_salarial = carregar(2, "p2_faixa-salarial.csv")

# Pergunta 3 - diversidade de gênero
p3_genero = carregar(3, "p3_genero.csv")
p3_faixa_salarial_genero = carregar(3, "p3_faixa-salarial-por-genero.csv")

# Pergunta 4 - tecnologias
p4_tecnologias = carregar(4, "p4_tecnologias.csv")

# Pergunta 5 - adoção de IA
p5_ia_prioridade = carregar(5, "p5_ia-prioridade.csv")

# Pergunta 6 - região, senioridade e modelo de trabalho
p6_adocao_tecnologia = carregar(6, "p6_adocao-tecnologia.csv")
p6_faixa_salarial_regiao = carregar(6, "p6_faixa-salaria-regial.csv")
p6_modelo_trabalho = carregar(6, "p6_modelo-trabalho.csv")

# Pergunta 7 - oportunidades e desafios
p7_fatores_retencao = carregar(7, "p7_fatores-retencao.csv")
p7_ia_desafios = carregar(7, "p7_ia-desafios-adocao.csv")

tabelas = {
    "p1_senioridade": p1_senioridade, "p1_setor": p1_setor, "p1_tamanho_time": p1_tamanho_time,
    "p2_faixa_salarial": p2_faixa_salarial,
    "p3_genero": p3_genero, "p3_faixa_salarial_genero": p3_faixa_salarial_genero,
    "p4_tecnologias": p4_tecnologias,
    "p5_ia_prioridade": p5_ia_prioridade,
    "p6_adocao_tecnologia": p6_adocao_tecnologia, "p6_faixa_salarial_regiao": p6_faixa_salarial_regiao,
    "p6_modelo_trabalho": p6_modelo_trabalho,
    "p7_fatores_retencao": p7_fatores_retencao, "p7_ia_desafios": p7_ia_desafios,
}
for nome, tabela in tabelas.items():
    print(f"{nome}: {tabela.shape[0]} linhas x {tabela.shape[1]} colunas")

## 2. Ajustes nos dados

Antes de gerar os gráficos, as tabelas analíticas passam por dois pequenos ajustes:

1. **Duas faixas salariais com erro de digitação são corrigidas** (`de R$ 101/mês a R$ 2.000/mês` → `de R$ 1.001/mês a R$ 2.000/mês`, e `de R$ 25.001/mês a R$ 3000/mês` → `de R$ 25.001/mês a R$ 30.000/mês`), presentes em `p2_faixa-salarial`, `p3_faixa-salarial-por-genero` e `p6_faixa-salaria-regial`. Como as duas variantes já existem como categorias separadas em algumas tabelas, os `groupby` usados neste notebook somam as duas automaticamente depois da correção.
2. **`modelo_trabalho` usa dois rótulos diferentes para "não informado"** — `"N/I"` e `"Não informado"` — que são unificados abaixo.

Do ponto de vista de arquitetura (Raw → Cleansed → Transformed → Curated), esses itens podem ser tratados no job PySpark da camada Transformed ou nas queries SQL do Athena. Como os dados já foram extraídos e publicados nas pastas locais para evitar reprocessamento em nuvem neste momento, esses pequenos tratamentos foram aplicados diretamente no notebook como ajustes de qualidade residuais.

Também é adotada aqui uma convenção usada em toda a análise: os percentuais deste notebook são sempre calculados **sobre as respostas válidas** de cada pergunta (excluindo "Não informado"/"N/I" do denominador), e não sobre o total de respondentes do ano — a base considerada é sempre indicada no texto. Isso é especialmente relevante para a pergunta sobre tamanho do time de dados, cuja taxa de resposta é baixa (ver Pergunta 1).

In [ ]:
# Corrige as duas faixas salariais com erro de digitação (ver item 1 acima).
CORRECOES_FAIXA_SALARIAL = {
    "de R$ 101/mês a R$ 2.000/mês": "de R$ 1.001/mês a R$ 2.000/mês",
    "de R$ 25.001/mês a R$ 3000/mês": "de R$ 25.001/mês a R$ 30.000/mês",
}

def corrigir_faixa_salarial(df, coluna="faixa_salarial"):
    # Corrige o rótulo da faixa e, quando existir, o valor numérico salario_inicial associado.
    df = df.copy()
    e_faixa_101 = df[coluna] == "de R$ 101/mês a R$ 2.000/mês"
    df[coluna] = df[coluna].replace(CORRECOES_FAIXA_SALARIAL)
    if "salario_inicial" in df.columns:
        df.loc[e_faixa_101, "salario_inicial"] = 1001
    return df

p2_faixa_salarial = corrigir_faixa_salarial(p2_faixa_salarial)
p3_faixa_salarial_genero = corrigir_faixa_salarial(p3_faixa_salarial_genero)
p6_faixa_salarial_regiao = corrigir_faixa_salarial(p6_faixa_salarial_regiao)

# Unifica os dois rótulos usados para "não informado" no modelo de trabalho (ver item 2 acima).
p6_modelo_trabalho["modelo_trabalho"] = p6_modelo_trabalho["modelo_trabalho"].replace({"N/I": "Não informado"})

print("Faixas salariais corrigidas e rótulos de 'não informado' unificados.")

## 3. Funções auxiliares

As tabelas analíticas já vêm com contagens (e, em vários casos, com o percentual) prontas por categoria a partir das consultas SQL no Athena. As funções abaixo apoiam três operações usadas em várias perguntas: recalcular um percentual apenas sobre respostas válidas, estimar uma mediana a partir de contagens por faixa (já que as queries agrupam os respondentes em faixas salariais/tamanhos) e desenhar o gráfico de barras horizontais usado em várias seções.

In [ ]:
# Ordem e ponto médio das faixas salariais, usados para aproximar estatísticas numéricas
# a partir das faixas (ver observação sobre essa aproximação na Pergunta 2).
FAIXAS_SALARIAIS_PONTO_MEDIO = {
    "Menos de R$ 1.000/mês": 500,
    "de R$ 1.001/mês a R$ 2.000/mês": 1500,
    "de R$ 2.001/mês a R$ 3.000/mês": 2500,
    "de R$ 3.001/mês a R$ 4.000/mês": 3500,
    "de R$ 4.001/mês a R$ 6.000/mês": 5000,
    "de R$ 6.001/mês a R$ 8.000/mês": 7000,
    "de R$ 8.001/mês a R$ 12.000/mês": 10000,
    "de R$ 12.001/mês a R$ 16.000/mês": 14000,
    "de R$ 16.001/mês a R$ 20.000/mês": 18000,
    "de R$ 20.001/mês a R$ 25.000/mês": 22500,
    "de R$ 25.001/mês a R$ 30.000/mês": 27500,
    "de R$ 30.001/mês a R$ 40.000/mês": 35000,
    "Acima de R$ 40.001/mês": 45000,
}

# Ordem e ponto médio das faixas de tamanho de time de dados.
NUM_PESSOAS_ORDEM = [
    "Ainda não temos pessoas atuando com dados na empresa",
    "1 - 3", "4 - 10", "11 - 20", "21 - 50", "51 - 100", "101 - 300",
    "Acima de 300 pessoas",
]
NUM_PESSOAS_PONTO_MEDIO = {
    "Ainda não temos pessoas atuando com dados na empresa": 0,
    "1 - 3": 2, "4 - 10": 7, "11 - 20": 15, "21 - 50": 35,
    "51 - 100": 75, "101 - 300": 200, "Acima de 300 pessoas": 350,
}

SENIORIDADE_ORDEM = ["Júnior", "Pleno", "Sênior", "Especialista/Staff+"]

NAO_INFORMADO = ("Não informado", "N/I")

def mediana_ponderada(valores, pesos):
    # Estima a mediana a partir de valores (ex.: pontos médios de faixa) e pesos (contagens por faixa).
    valores = np.asarray(valores, dtype=float)
    pesos = np.asarray(pesos, dtype=float)
    ordem = np.argsort(valores)
    valores_o, pesos_o = valores[ordem], pesos[ordem]
    acumulado = np.cumsum(pesos_o)
    alvo = acumulado[-1] / 2
    idx = np.searchsorted(acumulado, alvo)
    return valores_o[idx]

def distribuicao_valida(df, coluna_categoria, coluna_total, grupo=("ano_pesquisa",)):
    # Recalcula o percentual de cada categoria considerando apenas as respostas válidas do grupo
    # (exclui "Não informado"/"N/I" do denominador — ver convenção metodológica na Seção 2).
    grupo = list(grupo)
    validos = df[~df[coluna_categoria].isin(NAO_INFORMADO)].copy()
    total_valido = validos.groupby(grupo)[coluna_total].transform("sum")
    validos["pct"] = validos[coluna_total] / total_valido * 100
    return validos

def barh_serie_unica(serie, titulo, xlabel, ax=None, cor=COR_SEQUENCIAL, fmt="{:.0f}"):
    # Cria um gráfico de barras horizontais com os valores escritos.
    if ax is None:
        _, ax = plt.subplots(figsize=(8, max(2.5, 0.42 * len(serie))))
    serie = serie.sort_values()
    barras = ax.barh(serie.index.astype(str), serie.values, color=cor, height=0.62)
    ax.set_title(titulo, loc="left", pad=12)
    ax.set_xlabel(xlabel)
    ax.grid(axis="x", linewidth=0.7, alpha=0.6)
    ax.set_axisbelow(True)
    xmax = serie.values.max()
    ax.set_xlim(0, xmax * 1.18)
    for barra, valor in zip(barras, serie.values):
        ax.text(barra.get_width() + xmax * 0.015, barra.get_y() + barra.get_height() / 2,
                fmt.format(valor), va="center", ha="left", fontsize=9.5, color=TINTA_SECUNDARIA)
    return ax

print("Funções auxiliares prontas.")

---
## Pergunta 1 - Como está estruturado o mercado brasileiro de Dados?

Nesta etapa, são analisados o setor de atuação, a senioridade dos profissionais e o tamanho das equipes de Dados. Os resultados são comparados entre 2023 e 2025.

In [ ]:
# Top 10 setores que empregam profissionais de Dados em 2025 (% sobre respostas válidas).
setor_valido = distribuicao_valida(p1_setor, "setor", "total_profissionais")
top_setores = (
    setor_valido[setor_valido["ano_pesquisa"] == 2025]
    .set_index("setor")["pct"].sort_values(ascending=False).head(10)
)
barh_serie_unica(top_setores, "Top 10 setores que empregam profissionais de Dados (2025)",
                  "% dos respondentes com setor informado", fmt="{:.1f}%")

plt.tight_layout()
plt.show()

In [ ]:
# Distribuição por senioridade nos anos de 2023 a 2025 (% sobre respostas válidas de cargo/senioridade).
senioridade_ano = p1_senioridade.groupby(["ano_pesquisa", "senioridade"], as_index=False)["total_profissionais"].sum()
senioridade_ano = distribuicao_valida(senioridade_ano, "senioridade", "total_profissionais")
ordem_presente = [s for s in SENIORIDADE_ORDEM if s in senioridade_ano["senioridade"].unique()]

fig, ax = plt.subplots(figsize=(8, 4.5))
largura = 0.25
x = np.arange(len(ordem_presente))
for i, ano in enumerate(ANOS):
    valores = [senioridade_ano.query("ano_pesquisa == @ano and senioridade == @s")["pct"].sum() for s in ordem_presente]
    ax.bar(x + (i - 1) * largura, valores, width=largura, label=str(ano), color=COR_POR_ANO[ano])

ax.set_xticks(x, ordem_presente)
ax.set_ylabel("% dos respondentes no ano (base: cargo/senioridade informados)")
ax.set_title("Distribuição por senioridade - 2023 a 2025", loc="left", pad=12)
ax.grid(axis="y", linewidth=0.7, alpha=0.6)
ax.set_axisbelow(True)
ax.legend(title="Ano", frameon=False)

plt.tight_layout()
plt.show()

# Parcela de respondentes que não informaram cargo/senioridade, por ano — fora da base do gráfico acima.
nao_informado_senioridade = (
    p1_senioridade[p1_senioridade["senioridade"] == "Não informado"]
    .groupby("ano_pesquisa")["pct_do_ano"].sum()
)
print("% que não informou cargo/senioridade, por ano:")
print(nao_informado_senioridade)

In [ ]:
# Tamanho do time de dados na empresa - 2023 a 2025 (% sobre quem respondeu a essa pergunta).
time_valido = distribuicao_valida(p1_tamanho_time, "tamanho_time_dados", "total_empresas_respondentes")
ordem_presente = [t for t in NUM_PESSOAS_ORDEM if t in time_valido["tamanho_time_dados"].unique()]

fig, ax = plt.subplots(figsize=(10, 5.5))
altura = 0.22
y = np.arange(len(ordem_presente))

for i, ano in enumerate(ANOS):
    valores = [
        time_valido.query("ano_pesquisa == @ano and tamanho_time_dados == @t")["pct"].sum()
        for t in ordem_presente
    ]
    barras = ax.barh(y + (i - 1) * altura, valores, height=altura, label=str(ano), color=COR_POR_ANO[ano])
    for barra, valor in zip(barras, valores):
        ax.text(valor + 0.6, barra.get_y() + barra.get_height() / 2, f"{valor:.1f}%",
                va="center", ha="left", fontsize=8.5, color=TINTA_SECUNDARIA)

ax.set_yticks(y)
rotulos = [
    "Ainda não temos pessoas\natuando com dados na empresa" if t == "Ainda não temos pessoas atuando com dados na empresa" else t
    for t in ordem_presente
]
ax.set_yticklabels(rotulos)
ax.invert_yaxis()

ax.set_xlabel("% de quem respondeu essa pergunta no ano (base pequena — ver texto)")
ax.set_ylabel("")
ax.set_title("Tamanho do time de dados na empresa — 2023 a 2025", loc="left", pad=12)
ax.grid(axis="x", color=GRADE, linewidth=0.7, alpha=0.7)
ax.set_axisbelow(True)
ax.tick_params(axis="both", colors=TINTA_MUTED)

maior_valor = time_valido["pct"].max()
ax.set_xlim(0, maior_valor * 1.25)
ax.legend(title="Ano", frameon=False, loc="upper right", labelspacing=0.5)

plt.tight_layout()
plt.show()

base_valida = p1_tamanho_time[p1_tamanho_time["tamanho_time_dados"] != "Não informado"].groupby("ano_pesquisa")["total_empresas_respondentes"].sum()
base_total = p1_tamanho_time.groupby("ano_pesquisa")["total_empresas_respondentes"].sum()
print("Respostas válidas / total de respondentes do ano (taxa de resposta desta pergunta):")
print((base_valida / base_total * 100).round(1))

Entre os respondentes de 2025 com setor informado, Finanças/Bancos (18,5%) e Tecnologia (17,4%) concentram mais de um terço da amostra. A distribuição por senioridade muda pouco entre os anos: a maior mudança é o surgimento da categoria Especialista/Staff+ em 2025 (14,0% da base com cargo/senioridade informados), que reduz um pouco a participação relativa de Sênior e Pleno frente a 2023-2024 — mas cerca de 27-28% dos respondentes de cada ano não informou cargo/senioridade, e essa parcela fica de fora do gráfico de distribuição.

O tamanho das equipes de Dados é bastante diverso, mas a leitura mais importante é sobre a **base de resposta**: apenas entre 12% e 20% dos respondentes de cada ano respondeu a essa pergunta (896 de 5.293 em 2023, 1.045 de 5.215 em 2024, 652 de 3.494 em 2025), então os percentuais abaixo descrevem apenas essa fração da amostra, não o conjunto de respondentes do ano. Dentro dela, nenhuma faixa reúne a maioria: as equipes de até 10 pessoas somam entre 32,8% e 39,3% conforme o ano, e a participação de equipes grandes (acima de 300 pessoas) não cresce de forma constante — foi de 18,9% em 2023 para 23,8% em 2024 e caiu para 16,3% em 2025 — então não há, nesta amostra, evidência de uma tendência linear de expansão das equipes maiores.

---
## Pergunta 2 - Quais perfis profissionais são mais valorizados pelo mercado?

Os salários informados na pesquisa são coletados em faixas, não em valores exatos, e a tabela analítica (`p2_faixa-salarial`) reflete essa agregação: uma linha por combinação de ano, cargo, senioridade e faixa salarial, com a contagem de profissionais. Para comparar cargos e níveis numericamente, cada faixa é substituída pelo seu ponto médio (`FAIXAS_SALARIAIS_PONTO_MEDIO`) e a mediana é estimada a partir dessas contagens por faixa com `mediana_ponderada` — uma **aproximação** do salário real de cada respondente, e não o valor exato de cada pessoa (já que a query consolida os dados por faixa salarial).

Essa aproximação é particularmente imprecisa na faixa aberta "Acima de R$ 40.001/mês", cujo valor representativo (R$ 45.000) é arbitrado, já que a faixa não tem limite superior definido. Em 2025, essa faixa reúne uma parcela pequena da amostra (115 de 3.227 respostas válidas de salário, cerca de 3,6%).

A análise a seguir compara esse valor aproximado por cargo e por nível de senioridade.

In [ ]:
# Salário mediano por cargo em 2025 (cargos com pelo menos 30 respostas válidas de salário).
d2 = p2_faixa_salarial[
    (p2_faixa_salarial["ano_pesquisa"] == 2025)
    & (~p2_faixa_salarial["cargo"].isin(NAO_INFORMADO))
    & (~p2_faixa_salarial["faixa_salarial"].isin(NAO_INFORMADO))
].copy()
d2["ponto_medio"] = d2["faixa_salarial"].map(FAIXAS_SALARIAIS_PONTO_MEDIO)

salario_por_cargo = (
    d2.groupby("cargo")
    .apply(lambda x: pd.Series({
        "mediana": mediana_ponderada(x["ponto_medio"], x["total_profissionais"]),
        "n": x["total_profissionais"].sum(),
    }))
    .query("n >= 30")
    .sort_values("mediana")
)

# Dot plot — salário mediano por cargo
fig, ax = plt.subplots(figsize=(10, 6))
y = np.arange(len(salario_por_cargo))

ax.hlines(y=y, xmin=0, xmax=salario_por_cargo["mediana"], color=GRADE, linewidth=1.2, alpha=0.8, zorder=1)
ax.scatter(salario_por_cargo["mediana"], y, s=85, color=COR_COMPARACAO, zorder=3)

ax.set_yticks(y)
ax.set_yticklabels(salario_por_cargo.index, color=TINTA_MUTED)

for yi, valor in zip(y, salario_por_cargo["mediana"]):
    valor_formatado = f"R$ {valor:,.0f}".replace(",", ".")
    ax.text(valor + 350, yi, valor_formatado, va="center", ha="left", fontsize=9, color=TINTA_SECUNDARIA)

ax.set_title("Salário mediano por cargo — 2025", loc="left", pad=26)
ax.text(0, 1.01, "Cargos com pelo menos 30 respostas válidas de salário", transform=ax.transAxes,
        fontsize=9, color=TINTA_MUTED, ha="left", va="bottom")

ax.set_xticks([0, 5000, 10000, 15000, 20000])
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x / 1000:.0f}"))
ax.set_xlabel("Salário mediano mensal (R$ mil, aproximado)")
ax.set_ylabel("")

ax.grid(axis="x", color=GRADE, linewidth=0.7, alpha=0.7)
ax.set_axisbelow(True)
ax.tick_params(axis="both", colors=TINTA_MUTED)

maior_salario = salario_por_cargo["mediana"].max()
ax.set_xlim(0, maior_salario * 1.28)

plt.tight_layout()
plt.show()

In [ ]:
# Salário mediano por senioridade em 2025
d2b = p2_faixa_salarial[
    (p2_faixa_salarial["ano_pesquisa"] == 2025)
    & (~p2_faixa_salarial["senioridade"].isin(NAO_INFORMADO))
    & (~p2_faixa_salarial["faixa_salarial"].isin(NAO_INFORMADO))
].copy()
d2b["ponto_medio"] = d2b["faixa_salarial"].map(FAIXAS_SALARIAIS_PONTO_MEDIO)

salario_por_senioridade = (
    d2b.groupby("senioridade")
    .apply(lambda x: mediana_ponderada(x["ponto_medio"], x["total_profissionais"]))
    .reindex(SENIORIDADE_ORDEM).dropna()
)

fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(salario_por_senioridade.index, salario_por_senioridade.values, color=COR_SEQUENCIAL, width=0.55)

ax.set_title("Salário mediano por senioridade (2025)", loc="left", pad=12)
ax.set_ylabel("Salário mediano (R$/mês, aproximado)")
ax.grid(axis="y", linewidth=0.7, alpha=0.6)
ax.set_axisbelow(True)
ymax = salario_por_senioridade.values.max()
ax.set_ylim(0, ymax * 1.2)
for barra, valor in zip(barras, salario_por_senioridade.values):
    ax.text(barra.get_x() + barra.get_width() / 2, valor + ymax * 0.02, f"R$ {valor:,.0f}".replace(",", "."),
            ha="center", va="bottom", fontsize=9.5, color=TINTA_SECUNDARIA)

plt.tight_layout()
plt.show()

Entre os cargos com pelo menos 30 respostas em 2025, os maiores salários medianos aparecem em funções técnicas especializadas — Engenheiro de Machine Learning/AI Engineer (R$ 18.000) — e em Analytics Engineer e Data Product Manager (R$ 14.000), e não necessariamente em cargos de liderança/gestão. Cargos de entrada e de suporte, como Outra Opção, Analista de Suporte/Técnico e os papéis de Analista (Dados, BI, Negócios), aparecem entre os menores valores.

A senioridade também está associada ao salário: a mediana sobe de forma consistente de Júnior (R$ 3.500) a Especialista/Staff+ (R$ 18.000). Como essa última categoria só existe nos dados de 2025 (ver Pergunta 1), essa comparação é restrita a esse ano. O resultado é coerente com a leitura de que uma trajetória técnica avançada pode alcançar remuneração elevada mesmo sem exigir uma função de gestão, mas, por se tratar de dados de corte transversal (sem acompanhar as mesmas pessoas ao longo do tempo), não é possível afirmar causalidade entre especialização técnica e salário a partir apenas desta análise.

---
## Pergunta 3 - Qual é o cenário de diversidade de gênero nas carreiras de Dados?

Nesta parte, são analisados três pontos: a participação feminina ao longo dos anos, a participação por nível de senioridade em 2025 e a diferença salarial entre gêneros dentro de cada nível em 2025.

**Nota metodológica:** a evolução da participação feminina ao longo dos anos (gráfico abaixo) considera todas as respostas válidas da pergunta de gênero em `p3_genero`, incluindo as categorias "Outro" e "Prefiro não informar" (que são respostas legítimas à pergunta, não ausência de resposta). Já a comparação por senioridade e o gap salarial, calculados a partir de `p3_faixa-salarial-por-genero`, restringem a base a Feminino e Masculino, por serem as duas categorias com volume suficiente para uma comparação direta; a linha de referência da "média geral 2025" no gráfico de senioridade foi recalculada sobre essa mesma base binária, para ficar comparável com as barras.

In [ ]:
# Percentual de mulheres entre os respondentes de Dados (2023 a 2025)
pct_feminino = p3_genero[p3_genero["genero"] == "Feminino"].set_index("ano_pesquisa")["pct_genero"]

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(pct_feminino.index.astype(str), pct_feminino.values, marker="o", markersize=8,
        color=COR_DESTAQUE, linewidth=2.5)

for x, y in zip(pct_feminino.index.astype(str), pct_feminino.values):
    ax.text(x, y + 0.7, f"{y:.1f}%", ha="center", fontsize=10, color=TINTA_SECUNDARIA)

ax.set_title("% de mulheres entre os respondentes de Dados", loc="left", pad=12)
ax.set_ylabel("% do total de respondentes")
ax.set_ylim(0, max(pct_feminino.values) * 1.4)
ax.grid(axis="y", linewidth=0.7, alpha=0.6)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Percentual de mulheres por senioridade em 2025 (base: Feminino + Masculino, com senioridade informada)
d3 = p3_faixa_salarial_genero[
    (p3_faixa_salarial_genero["ano_pesquisa"] == 2025)
    & (~p3_faixa_salarial_genero["senioridade"].isin(NAO_INFORMADO))
]
por_senioridade = d3.groupby(["senioridade", "genero"])["total_respondentes"].sum().unstack("genero")
pct_fem_senioridade = (
    (por_senioridade["Feminino"] / (por_senioridade["Feminino"] + por_senioridade["Masculino"]) * 100)
    .reindex(SENIORIDADE_ORDEM).dropna()
)

total_fem = por_senioridade["Feminino"].sum()
total_masc = por_senioridade["Masculino"].sum()
media_2025 = total_fem / (total_fem + total_masc) * 100

fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(pct_fem_senioridade.index, pct_fem_senioridade.values, color=COR_DESTAQUE, width=0.55)

ax.axhline(media_2025, color=TINTA_MUTED, linestyle="--", linewidth=1.2, zorder=1)
ax.text(0.01, media_2025 + 0.35, f"geral 2025 ({media_2025:.1f}%)", transform=ax.get_yaxis_transform(),
        fontsize=9, color=TINTA_MUTED, ha="left", va="bottom",
        bbox=dict(facecolor=SUPERFICIE, edgecolor="none", pad=1.5), zorder=3)

ax.set_title("% de mulheres por senioridade (2025)", loc="left", pad=12)
ax.set_ylabel("% mulheres dentro do nível (base: Fem. + Masc.)")
ax.grid(axis="y", color=GRADE, linewidth=0.7, alpha=0.7)
ax.set_axisbelow(True)

for barra, valor in zip(barras, pct_fem_senioridade.values):
    ax.text(barra.get_x() + barra.get_width() / 2, valor + 0.4, f"{valor:.1f}%",
            ha="center", va="bottom", fontsize=9.5, color=TINTA_SECUNDARIA)

plt.tight_layout()
plt.show()

In [ ]:
# Salário mediano por gênero e senioridade em 2025
d3b = p3_faixa_salarial_genero[
    (p3_faixa_salarial_genero["ano_pesquisa"] == 2025)
    & (~p3_faixa_salarial_genero["senioridade"].isin(NAO_INFORMADO))
    & (~p3_faixa_salarial_genero["faixa_salarial"].isin(NAO_INFORMADO))
].copy()
d3b["ponto_medio"] = d3b["faixa_salarial"].map(FAIXAS_SALARIAIS_PONTO_MEDIO)

gap_salarial = (
    d3b.groupby(["senioridade", "genero"])
    .apply(lambda x: mediana_ponderada(x["ponto_medio"], x["total_respondentes"]))
    .unstack("genero").reindex(SENIORIDADE_ORDEM).dropna()
)
# O gap é calculado sobre a mediana do ponto médio das faixas salariais
# (aproximação — ver Pergunta 2), não sobre o salário exato de cada pessoa.
gap_salarial["gap_%"] = (1 - gap_salarial["Feminino"] / gap_salarial["Masculino"]) * 100

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(gap_salarial))
largura = 0.35

ax.bar(x - largura / 2, gap_salarial["Masculino"], width=largura, label="Masculino", color=COR_SEC_1)
ax.bar(x + largura / 2, gap_salarial["Feminino"], width=largura, label="Feminino", color=COR_DESTAQUE)
ax.set_xticks(x, gap_salarial.index)
ax.set_ylabel("Salário mediano (R$/mês, aproximado)")
ax.set_title("Salário mediano por gênero e senioridade (2025)", loc="left", pad=12)
ax.grid(axis="y", linewidth=0.7, alpha=0.6)
ax.set_axisbelow(True)
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

gap_salarial.round(0)

A participação feminina entre os respondentes caiu de 24,4% em 2023 para 22,0% em 2025. Em 2025, considerando apenas Feminino e Masculino, a participação feminina também varia por senioridade: 28,5% entre os Júnior, caindo para 22,4% em Pleno, 20,9% em Sênior e 20,2% em Especialista/Staff+ — sempre abaixo da média geral de 22,8%. Como 2023, 2024 e 2025 são amostras distintas e os grupos de diferentes senioridades são compostos por respondentes diferentes, não pelas mesmas pessoas acompanhadas ao longo do tempo (ver Limitações da análise), esse resultado é melhor descrito como um gradiente de representatividade por senioridade, com menor participação feminina nas categorias mais altas — e não como evidência de um funil de carreira que acompanhe as mesmas pessoas, nem como prova de causa entre senioridade e sub-representação feminina.

A comparação salarial de 2025 não mostra diferença na mediana entre homens e mulheres em Júnior e Pleno, mas mostra um gap relevante em Sênior (28,6%: R$ 10.000 para mulheres vs. R$ 14.000 para homens) e em Especialista/Staff+ (22,2%: R$ 14.000 vs. R$ 18.000), calculado sobre o ponto médio das faixas salariais. Esses dois resultados merecem atenção particular, mas — pelas mesmas razões apontadas acima — devem ser tratados como evidência descritiva da amostra, e não como relação causal estabelecida.

---
## Pergunta 4 - Quais tecnologias apresentam maior adoção entre os profissionais?

`p4_tecnologias` já traz, por ano e tecnologia, o percentual de adoção calculado sobre as respostas válidas daquela tecnologia (`pct_adocao`) — quando uma tecnologia não foi perguntada em determinado ano (por exemplo, DAX antes de 2025, ou JavaScript em 2025), a linha correspondente vem com `total_respostas_validas = 0` e `pct_adocao` vazio, e é descartada com `dropna`.

As tecnologias são organizadas de acordo com o percentual de uso em 2025. Em seguida, a adoção de AWS, Google Cloud e Azure é comparada entre os três anos.

In [ ]:
# Top 15 tecnologias mais usadas no trabalho em 2025
adocao_2025 = (
    p4_tecnologias[p4_tecnologias["ano_pesquisa"] == 2025]
    .dropna(subset=["pct_adocao"])
    .set_index("tecnologia")["pct_adocao"]
    .sort_values(ascending=False).head(15)
)
barh_serie_unica(adocao_2025.sort_values(), "Top 15 tecnologias mais usadas no trabalho (2025)",
                  "% dos profissionais que usam (sobre respostas válidas)", fmt="{:.1f}%")

plt.tight_layout()
plt.show()

In [ ]:
# Adoção de provedores de nuvem - 2023 a 2025
clouds = ["AWS", "GCP", "Azure"]
NOMES_CLOUD = {"AWS": "AWS", "GCP": "Google Cloud", "Azure": "Azure"}
cores_cloud = {"AWS": COR_COMPARACAO, "Google Cloud": COR_SEC_1, "Azure": COR_DESTAQUE}

adocao_cloud = (
    p4_tecnologias[p4_tecnologias["tecnologia"].isin(clouds)]
    .pivot(index="ano_pesquisa", columns="tecnologia", values="pct_adocao")
    .rename(columns=NOMES_CLOUD)
    .sort_index()
)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
anos_idx = adocao_cloud.index.astype(int)

# Deslocamentos verticais (em pontos) dos rótulos de 2023/2024, ajustados para afastar rótulos de séries
# que ficam próximas naquele ano (AWS/Google Cloud em 2023, Azure/Google Cloud em 2024).
offsets = {
    ("AWS", 2023): -14, ("Google Cloud", 2023): 14, ("Azure", 2023): 12,
    ("AWS", 2024): 12, ("Azure", 2024): 14, ("Google Cloud", 2024): -14,
}

for cloud in NOMES_CLOUD.values():
    valores = adocao_cloud[cloud]
    ax.plot(anos_idx, valores, marker="o", markersize=7, linewidth=2.4, color=cores_cloud[cloud], zorder=3)

    for ano in anos_idx[:-1]:
        valor = adocao_cloud.loc[ano, cloud]
        dy = offsets[(cloud, ano)]
        ax.annotate(f"{valor:.1f}%", xy=(ano, valor), xytext=(0, dy),
                    textcoords="offset points", ha="center", va="center", fontsize=8.5,
                    color=cores_cloud[cloud], zorder=5)

    valor_final = valores.iloc[-1]
    ax.annotate(f"{cloud}  {valor_final:.1f}%", xy=(anos_idx[-1], valor_final), xytext=(12, 0),
                textcoords="offset points", ha="left", va="center", fontsize=9.5,
                fontweight="medium", color=cores_cloud[cloud], zorder=5)

ax.set_title("Adoção de provedores de nuvem — 2023 a 2025", loc="left", pad=12)
ax.set_ylabel("% dos profissionais que usam (sobre respostas válidas)")
ax.set_xticks(anos_idx)
ax.set_xlim(anos_idx.min() - 0.3, anos_idx.max() + 0.9)
ax.grid(axis="y", linewidth=0.7, alpha=0.6)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

SQL (98,1%), Python (94,3%) e Java (85,3%) lideram as tecnologias mais citadas em 2025 — Java aparece com uma base de respostas válidas bem menor nesse ano (566, ante 3.772 em 2023), o que é coerente com uma pergunta reformulada/condicional na edição de 2025, então sua posição no ranking deve ser lida com essa ressalva. Entre as ferramentas com base de resposta mais estável ao longo dos anos, Power BI (71,6%) segue como a ferramenta de BI mais citada.

Entre os provedores de nuvem, houve mudança de liderança ao longo do período: em 2023 a Azure era o provedor mais citado (42,3%), à frente de Google Cloud (31,6%) e AWS (30,6%); a partir de 2024 a AWS assumiu a liderança (44,1%) e ampliou a distância até 2025 (48,3% AWS vs. 34,8% Azure vs. 30,9% Google Cloud).

---
## Pergunta 5 - Qual é o nível de adoção de Inteligência Artificial e qual é o seu impacto?

A tabela analítica disponível para esta pergunta (`p5_ia-prioridade`) traz, por ano, a parcela de empresas para quem IA Generativa não é prioridade e a parcela para quem ela já é a principal frente do negócio — ambas calculadas sobre as respostas válidas da pergunta de prioridade estratégica de IA.


In [ ]:
# Prioridade estratégica de IA na empresa - 2023 a 2025
prioridade = p5_ia_prioridade.pivot(index="ano_pesquisa", columns="indicador", values="pct_sim")
base = p5_ia_prioridade.pivot(index="ano_pesquisa", columns="indicador", values="total_respostas_validas")

# Gráfico de barras comparando os dois indicadores de prioridade estratégica de IA.
indicadores = ["IA é prioridade como principal frente", "IA não é prioridade na empresa"]
cores_indicador = {indicadores[0]: COR_COMPARACAO, indicadores[1]: COR_SEC_2}

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(prioridade.index))
largura = 0.35

# Desloca as barras de cada indicador para que fiquem lado a lado, e escreve os valores acima das barras.
for i, indicador in enumerate(indicadores):
    valores = prioridade[indicador].values
    deslocamento = (i - 0.5) * largura
    barras = ax.bar(x + deslocamento, valores, width=largura, label=indicador, color=cores_indicador[indicador])
    for barra, valor in zip(barras, valores):
        ax.text(barra.get_x() + barra.get_width() / 2, valor + 0.6, f"{valor:.1f}%",
                ha="center", va="bottom", fontsize=9, color=TINTA_SECUNDARIA)

# Configurações do gráfico
ax.set_xticks(x, prioridade.index.astype(str))
ax.set_ylabel("% das respostas válidas sobre prioridade de IA")
ax.set_title("Prioridade estratégica de IA Generativa na empresa — 2023 a 2025", loc="left", pad=12)
ax.grid(axis="y", linewidth=0.7, alpha=0.6)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=1)

plt.tight_layout()
plt.show()

print("base de respostas válidas por ano:", base[indicadores[0]].to_dict())

A parcela de empresas para quem IA Generativa não é prioridade caiu de forma acentuada, de 20,8% em 2023 para 6,6% em 2025. Ao mesmo tempo, a parcela que já a trata como principal frente do negócio também recuou, de 25,1% para 17,4% — sugerindo que o movimento não é simplesmente "de não-prioridade para prioridade máxima", mas uma migração de empresas para categorias intermediárias de priorização (não capturadas por este indicador binário). A base de respostas válidas também caiu de forma expressiva em 2025 (2.721, ante ~4.630 nos dois anos anteriores), acompanhando a redução do tamanho da amostra total daquele ano — então a mudança percentual deve ser lida com essa ressalva de base.

---
## Pergunta 6 - Existem diferenças relevantes entre regiões, senioridades ou modelos de trabalho?

Nesta etapa, são comparados o salário por região, a distribuição dos modelos de trabalho ao longo dos três anos (2023 a 2025) e a adoção de tecnologias avançadas por senioridade.

In [ ]:
# Salário médio por região em 2025
d6 = p6_faixa_salarial_regiao[
    (p6_faixa_salarial_regiao["ano_pesquisa"] == 2025)
    & (~p6_faixa_salarial_regiao["regiao"].isin(NAO_INFORMADO))
    & (~p6_faixa_salarial_regiao["faixa_salarial"].isin(NAO_INFORMADO))
].copy()
d6["ponto_medio"] = d6["faixa_salarial"].map(FAIXAS_SALARIAIS_PONTO_MEDIO)

# Calcula o salário médio ponderado pelo número de profissionais em cada faixa salarial.
salario_por_regiao = (
    d6.groupby("regiao")
    .apply(lambda x: pd.Series({
        "media": np.average(x["ponto_medio"], weights=x["total_profissionais"]),
        "n": x["total_profissionais"].sum(),
    }))
    .sort_values("media")
)

# Dot plot — salário médio por região
fig, ax = plt.subplots(figsize=(8.5, 4.5))
y = np.arange(len(salario_por_regiao))

ax.hlines(y=y, xmin=0, xmax=salario_por_regiao["media"], color=COR_SEC_2, linewidth=1.7, alpha=0.65, zorder=1)
ax.scatter(salario_por_regiao["media"], y, s=85, color=COR_COMPARACAO, zorder=3)

ax.set_yticks(y)
ax.set_yticklabels(salario_por_regiao.index, color=TINTA_MUTED)

# Escreve os valores de salário médio ao lado de cada ponto, formatando como R$ e com separador de milhar.
for yi, valor in zip(y, salario_por_regiao["media"]):
    valor_formatado = f"R$ {valor:,.0f}".replace(",", ".")
    ax.annotate(valor_formatado, xy=(valor, yi), xytext=(8, 0), textcoords="offset points",
                ha="left", va="center", fontsize=9, color=TINTA_SECUNDARIA, zorder=5)

ax.set_title("Salário médio por região — 2025", loc="left", pad=26)
ax.text(0, 1.01, "Média do ponto médio das faixas salariais", transform=ax.transAxes,
        fontsize=9, color=TINTA_MUTED, ha="left", va="bottom")

ax.set_xticks([0, 5000, 10000, 15000])
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x / 1000:.0f}"))

ax.set_xlabel("Salário médio mensal (R$ mil, aproximado)")
ax.set_ylabel("")

maior_salario = salario_por_regiao["media"].max()

ax.set_xlim(0, max(15000, maior_salario * 1.18))
ax.grid(axis="x", color=GRADE, linewidth=0.7, alpha=0.5)
ax.set_axisbelow(True)
ax.tick_params(axis="both", colors=TINTA_MUTED)

plt.tight_layout()
plt.show()

print(salario_por_regiao)

In [ ]:
# Modelo de trabalho atual — 2023 a 2025
modelo_valido = distribuicao_valida(
    p6_modelo_trabalho,
    "modelo_trabalho",
    "total_profissionais"
)

anos_disponiveis = [2023, 2024, 2025]
modelos = modelo_valido["modelo_trabalho"].unique().tolist()

# Do mais remoto para o mais presencial
ordem_modelo = sorted(
    modelos,
    key=lambda m: (
        "remoto" not in m.lower(),
        "híbrido" not in m.lower(),
        m
    )
)

# Cria um gráfico de barras empilhadas para mostrar a evolução do modelo de trabalho nos anos disponíveis.
def cor_modelo(modelo):
    texto = modelo.lower()

    if "100% remoto" in texto:
        return COR_COMPARACAO

    if "híbrido" in texto and "flex" in texto:
        return COR_DESTAQUE

    if "híbrido" in texto:
        return COR_SEC_1

    return COR_SEC_3


def rotulo_modelo(modelo):
    texto = modelo.lower()

    if "100% remoto" in texto:
        return "100% remoto"

    if "híbrido" in texto and "flex" in texto:
        return "Híbrido — flexível"

    if "híbrido" in texto:
        return "Híbrido — dias fixos"

    if "100% presencial" in texto:
        return "100% presencial"

    return modelo


fig, ax = plt.subplots(figsize=(10, 4.8))

bottom = np.zeros(len(anos_disponiveis))

# Cria as barras empilhadas para cada modelo de trabalho, calculando os valores percentuais para cada ano.
for modelo in ordem_modelo:
    valores = [
        modelo_valido.query(
            "ano_pesquisa == @ano and modelo_trabalho == @modelo"
        )["pct"].sum()
        for ano in anos_disponiveis
    ]

    barras = ax.bar(
        [str(a) for a in anos_disponiveis],
        valores,
        bottom=bottom,
        width=0.5,
        color=cor_modelo(modelo),
        label=rotulo_modelo(modelo)
    )

    for i, (barra, valor) in enumerate(zip(barras, valores)):
        if valor >= 6:
            ax.text(
                barra.get_x() + barra.get_width() / 2,
                bottom[i] + valor / 2,
                f"{valor:.1f}%",
                ha="center",
                va="center",
                fontsize=9,
                fontweight="medium",
                color="white" if valor >= 12 else TINTA_PRIMARIA
            )

    bottom += np.array(valores)

ax.set_title(
    "Evolução do modelo de trabalho atual — 2023 a 2025",
    loc="left",
    pad=24
)

# Configurações de estilo do gráfico
ax.set_ylabel("% dos respondentes (respostas válidas)")
ax.set_xlabel("")

ax.set_ylim(0, 100)
ax.set_yticks(np.arange(0, 101, 20))

ax.grid(
    axis="y",
    color=GRADE,
    linewidth=0.7,
    alpha=0.6
)

ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color(GRADE)

ax.tick_params(
    axis="both",
    colors=TINTA_MUTED
)

ax.legend(
    title="Modelo de trabalho",
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    labelspacing=0.8
)

plt.tight_layout(rect=[0, 0, 0.78, 1])
plt.show()

In [ ]:
# Adoção de tecnologias avançadas por senioridade em 2025
d6c = p6_adocao_tecnologia[
    (p6_adocao_tecnologia["ano_pesquisa"] == 2025)
    & (~p6_adocao_tecnologia["senioridade"].isin(NAO_INFORMADO))
].set_index("senioridade").reindex(SENIORIDADE_ORDEM).dropna()

# Dicionário para mapear os nomes das tecnologias para exibição no gráfico.
NOMES_TECNOLOGIA = {
    "pct_cloud": "Cloud (qualquer provedor)",
    "pct_databricks": "Databricks",
    "pct_snowflake": "Snowflake",
    "pct_airflow": "Airflow",
}
cores_tecnologia = {
    "Cloud (qualquer provedor)": COR_COMPARACAO,
    "Databricks": COR_DESTAQUE,
    "Snowflake": COR_SEC_1,
    "Airflow": COR_SEC_2,
}
offsets = {"Cloud (qualquer provedor)": 9, "Databricks": 9, "Snowflake": -12, "Airflow": 9}

# Gráfico de linhas mostrando a adoção de tecnologias avançadas por senioridade em 2025.
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(d6c.index))

# Plota cada tecnologia como uma linha no gráfico, com marcadores e anotações de valores.
for coluna, nome in NOMES_TECNOLOGIA.items():
    valores = d6c[coluna]
    ax.plot(x, valores, marker="o", markersize=7, linewidth=2.4, color=cores_tecnologia[nome], zorder=3)
    for xi, valor in zip(x[:-1], valores.iloc[:-1]):
        ax.annotate(f"{valor:.1f}%", xy=(xi, valor), xytext=(0, offsets[nome]), textcoords="offset points",
                    ha="center", va="center", fontsize=8.5, color=cores_tecnologia[nome], zorder=5)
    valor_final = valores.iloc[-1]
    ax.annotate(f"{nome}  {valor_final:.1f}%", xy=(x[-1], valor_final), xytext=(12, 0),
                textcoords="offset points", ha="left", va="center", fontsize=9.5,
                fontweight="medium", color=cores_tecnologia[nome], zorder=5)

ax.set_title("Adoção de tecnologias por senioridade — 2025", loc="left", pad=12)
ax.set_ylabel("% dos respondentes do nível que usam a tecnologia")
ax.set_xticks(x, d6c.index)
ax.set_xlim(-0.15, len(x) - 1 + 0.70)

ax.grid(axis="y", linewidth=0.7, alpha=0.6)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

Os dados sugerem diferenças regionais nos salários médios de 2025 entre os respondentes: a região Sudeste apresenta a maior média (R$ 13.542), enquanto o Nordeste apresenta a menor (R$ 10.330). A região Norte tem apenas 36 respostas válidas — muito menos que o Sudeste (2.024) —, por isso seu resultado (R$ 12.681) deve ser interpretado com cautela e não tratado como representativo da região.

**Evolução dos modelos de trabalho (2023 a 2025):**
O trabalho 100% remoto manteve-se hegemônico nos primeiros dois anos (46,3% em 2023 e 45,7% em 2024), mas recuou de forma nítida em 2025 para 39,7% (-6,6 p.p. no triênio). Em contrapartida, o modelo 100% presencial, que vinha estável em 2023 (16,6%) e 2024 (16,3%), subiu para 20,8% em 2025 (+4,5 p.p.). Os modelos híbridos somados (flexível e com dias fixos) mantiveram trajetória de ligeiro avanço contínuo, subindo de 37,1% em 2023 para 38,0% em 2024 e atingindo 39,5% em 2025 — praticamente empatando em relevância com o trabalho 100% remoto.

**Adoção de tecnologias por senioridade:**
Em 2025, a adoção de tecnologias avançadas cresce com a senioridade para Cloud, Snowflake e Airflow, sugerindo espaço para capacitar profissionais Júnior e Pleno nessas ferramentas. O Databricks é uma exceção parcial: a adoção sobe até o nível Sênior (37,3%) e cai entre os Especialistas/Staff+ (31,0%), o que pode refletir diferenças de stack tecnológico por tipo de função nesse grupo — mais avançado tecnicamente, porém mais heterogêneo — e não necessariamente um menor domínio da ferramenta. Vale notar que a métrica `pct_cloud` desta tabela mede o uso de qualquer provedor de nuvem combinado, não de um provedor específico.

---
## Pergunta 7 - Quais oportunidades e desafios podem ser identificados para empresas que desejam investir em Dados e Inteligência Artificial?

Para concluir, são analisados os principais desafios para adoção de IA Generativa nas empresas, os fatores mais relevantes para reter profissionais de Dados e a evolução do tamanho das equipes de Dados. As tabelas `p7_ia-desafios-adocao` e `p7_fatores-retencao` cobrem os três anos da pesquisa (2023-2025).

In [ ]:
# Principais desafios para adoção de IA Generativa nas empresas (2025)
desafios_2025 = (
    p7_ia_desafios[p7_ia_desafios["ano_pesquisa"] == 2025]
    .set_index("desafio")["pct_empresas_afetadas"].sort_values()
)
barh_serie_unica(desafios_2025, "Principais desafios para adoção de IA Generativa nas empresas (2025)",
                  "% dos respondentes elegíveis que apontam o desafio", cor=COR_DESTAQUE, fmt="{:.1f}%")

plt.tight_layout()
plt.show()

# Evolução 2023 -> 2025 dos desafios com maior variação.
evolucao = p7_ia_desafios.pivot(index="desafio", columns="ano_pesquisa", values="pct_empresas_afetadas")
evolucao["variacao_pp"] = evolucao[2025] - evolucao[2023]

print(evolucao.sort_values("variacao_pp"))

In [ ]:
# Fatores mais relevantes para reter profissionais de Dados (2025)
fatores_2025 = (
    p7_fatores_retencao[p7_fatores_retencao["ano_pesquisa"] == 2025]
    .set_index("fator")["pct_fator_relevante"].sort_values()
)
barh_serie_unica(fatores_2025, "Fatores mais relevantes para reter profissionais de Dados (2025)",
                  "% dos respondentes que apontam o fator", cor=COR_SEQUENCIAL, fmt="{:.1f}%")

plt.tight_layout()
plt.show()

In [ ]:
# Tamanho do time de dados e % de empresas ainda sem time de dados — 2023 a 2025
d1 = p1_tamanho_time.copy()
d1["ponto_medio"] = d1["tamanho_time_dados"].map(NUM_PESSOAS_PONTO_MEDIO)
d1v = d1.dropna(subset=["ponto_medio"])

# Calcula o tamanho mediano do time de dados por ano, ponderado pelo número de empresas respondentes.
tamanho_mediano_ano = d1v.groupby("ano_pesquisa").apply(
    lambda x: mediana_ponderada(x["ponto_medio"], x["total_empresas_respondentes"])
)

# Calcula o percentual de empresas que ainda não têm pessoas atuando com dados, por ano.
sem_time_valido = distribuicao_valida(p1_tamanho_time, "tamanho_time_dados", "total_empresas_respondentes")
pct_sem_time = (
    sem_time_valido[sem_time_valido["tamanho_time_dados"] == "Ainda não temos pessoas atuando com dados na empresa"]
    .set_index("ano_pesquisa")["pct"]
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))

# Gráfico de linha para o tamanho mediano do time de dados
axes[0].plot(tamanho_mediano_ano.index.astype(str), tamanho_mediano_ano.values, marker="o",
             color=COR_SEQUENCIAL, linewidth=2.4, markersize=8)
axes[0].set_title("Tamanho mediano do time de dados", loc="left", pad=10)
axes[0].set_ylabel("Nº de pessoas (ponto médio da faixa)")
axes[0].grid(axis="y", linewidth=0.7, alpha=0.6)
axes[0].set_axisbelow(True)

axes[1].bar(pct_sem_time.index.astype(str), pct_sem_time.values, color=COR_DESTAQUE, width=0.5)
axes[1].set_title("% de empresas ainda sem time de dados", loc="left", pad=10)
axes[1].set_ylabel("% de quem respondeu essa pergunta no ano")
axes[1].set_ylim(0, pct_sem_time.values.max() * 1.35)
axes[1].grid(axis="y", linewidth=0.7, alpha=0.6)
axes[1].set_axisbelow(True)

# Adiciona os valores percentuais acima das barras do gráfico de % de empresas sem time de dados
for x, y in zip(pct_sem_time.index.astype(str), pct_sem_time.values):
    axes[1].text(x, y + pct_sem_time.values.max() * 0.04, f"{y:.1f}%", ha="center", fontsize=9.5, color=TINTA_SECUNDARIA)

plt.tight_layout()
plt.show()

print("respostas válidas sobre tamanho de time, por ano:",
      p1_tamanho_time[p1_tamanho_time["tamanho_time_dados"] != "Não informado"].groupby("ano_pesquisa")["total_empresas_respondentes"].sum().to_dict())

**Desafios:** com os dados agora cobrindo os três anos, dá para ver que os desafios mais citados em 2025 — falta de expertise/recursos (38,8%) e dados da empresa não estarem prontos (36,8%) — também foram os que mais cresceram desde 2023 (+4,1 p.p. e +6,2 p.p., respectivamente). Em contrapartida, incerteza regulatória (-4,6 p.p.) e preocupação com propriedade intelectual (-5,5 p.p.) perderam peso relativo no período. ROI não comprovado teve pico em 2024 (39,9%) e recuou em 2025 (32,8%), mas segue entre os itens mais citados nos três anos. Como a pergunta é condicional — exibida apenas a quem ainda não usa (ou usa pouco) IA Generativa na empresa —, a base é sempre uma fração menor da amostra total do ano (587 respondentes em 2025).

Os resultados reforçam que a preparação dos dados (qualidade, governança e organização) e a capacitação técnica das equipes são pré-condições relevantes, observadas nesta amostra, antes de ampliar projetos de IA — sem que isso implique que resolvê-las seja suficiente para garantir adoção bem-sucedida.

**Retenção:** salário/remuneração é, disparadamente, o fator mais citado para reter profissionais de Dados nos três anos (81,4% a 84,2%), seguido por flexibilidade/trabalho remoto (estável entre 58% e 59%), plano de carreira/crescimento (32% a 34%) e benefícios (26% a 28%). A estabilidade desse ranking ao longo dos três anos sugere que, ao menos nesta amostra, as prioridades de retenção dos profissionais de Dados não mudaram de forma relevante no período — o que é uma leitura direta e de baixa ambiguidade, diferente da maioria das demais perguntas deste notebook.

**Oportunidades:** o tamanho mediano das equipes de Dados varia entre os anos (15, 35 e 15 pessoas em 2023, 2024 e 2025, respectivamente) e é calculado sobre uma fração pequena da amostra em cada ano (entre 652 e 1.045 respostas, de um total de milhares) — por isso essa série deve ser interpretada com cautela, sem inferir uma tendência. A grande maioria das empresas que respondeu à pergunta já possui alguma estrutura de Dados: a parcela sem nenhuma pessoa dedicada ficou entre 4,9% e 6,9% ao longo dos anos.

Combinadas com as demais perguntas deste notebook, as evidências apontam para oportunidades concentradas em quatro frentes: elevar a maturidade e a governança dos dados, capacitar profissionais Júnior e Pleno nas tecnologias mais avançadas (Pergunta 6), transformar a redução da não-priorização de IA (Pergunta 5) em iniciativas de negócio mais estruturadas, e manter salário e flexibilidade como pilares de retenção — o par mais citado, e de forma consistente, nos três anos.

---
## Limitações da análise

Antes da conclusão, iremos explicitar os principais limites metodológicos das análises anteriores:

- **Os resultados descrevem os respondentes da pesquisa, não o mercado como um todo.** Os dados analisados não constituem, neste trabalho, uma amostra probabilística do mercado brasileiro de Dados. Portanto, os resultados devem ser interpretados como características dos respondentes da pesquisa, e não como estimativas populacionais do mercado como um todo.

- **Os três anos não formam um painel longitudinal.** 2023, 2024 e 2025 são amostras distintas, sem garantia de que os mesmos profissionais responderam em mais de uma edição. Variações entre anos podem refletir mudanças reais no mercado, mas também mudanças em quem respondeu à pesquisa em cada ano.

- **Os salários são faixas, não valores exatos.** Todas as análises salariais usam o ponto médio da faixa informada como aproximação, o que é especialmente impreciso na faixa aberta "Acima de R$ 40.001/mês" (ver Pergunta 2). Como as tabelas resultantes das queries vêm agrupadas por faixa salarial (e não como valor salarial individual pontual), essa aproximação é feita a partir de contagens por faixa (`mediana_ponderada`), não de valores individuais.

- **O tamanho das equipes também é informado em faixas**, com a mesma lógica de ponto médio, e tem taxa de resposta baixa nas três edições (entre 12% e 20% dos respondentes do ano — ver Pergunta 1). As medianas apresentadas descrevem apenas quem respondeu a essa pergunta.

- **Algumas perguntas têm amostras menores ou cobertura mais restrita.** Perguntas condicionais (por exemplo, os desafios de adoção de IA Generativa) só são exibidas a um subconjunto de respondentes; e a Pergunta 5 conta com um indicador binário de prioridade de IA, sem capturar níveis intermediários de adoção ou medir diretamente o impacto financeiro no negócio. Cada uma dessas restrições foi indicada no texto da seção correspondente.

- **Diferenças observadas não implicam causalidade.** Associações entre senioridade, gênero, tecnologia e salário são descritivas da amostra em cada ano, e não evidência de relação causal entre essas variáveis.

- **As análises são descritivas.** Não foram realizados testes de hipótese, intervalos de confiança ou outros testes inferenciais para verificar significância estatística das diferenças entre anos ou grupos. Portanto, especialmente diferenças pequenas devem ser interpretadas como variações observadas na amostra, e não necessariamente como diferenças estatisticamente significativas na população.

---
## Conclusão

Ao reunir as tabelas das três edições da pesquisa State of Data Brasil processadas pelo pipeline e pelas consultas analíticas no Athena, este notebook traça um retrato do mercado brasileiro de Dados entre 2023 e 2025 — sempre com a ressalva de que se trata de um retrato da amostra de respondentes, não do mercado como um todo (ver Limitações da análise).

**Estrutura do mercado.** Entre os respondentes de 2025 com setor informado, Finanças/Bancos e Tecnologia concentram mais de um terço da amostra, mas o tamanho das equipes de Dados é heterogêneo entre quem respondeu a essa pergunta: não há uma faixa dominante, e a participação de equipes grandes oscila entre os anos, sem tendência clara de crescimento. A distribuição por senioridade é relativamente estável, com maioria de profissionais Plenos e Sêniores, ainda que cerca de um quarto dos respondentes não informe cargo/senioridade em cada ano.

**Valorização de especialização e senioridade.** Nesta amostra, os maiores salários medianos de 2025 aparecem menos em cargos de liderança do que em funções técnicas especializadas (Machine Learning Engineer, Analytics Engineer) e em produto (Data Product Manager). A senioridade também acompanha o salário de forma consistente, e a categoria Especialista/Staff+ — que só passou a existir na pesquisa em 2025 — reforça que uma trajetória técnica avançada pode alcançar remuneração elevada mesmo sem exigir uma função de gestão.

**Diversidade de gênero.** A participação feminina caiu de 24,4% para 22,0% dos respondentes entre 2023 e 2025 e é menor nos níveis de maior senioridade (de 28,5% em Júnior a 20,2% em Especialista/Staff+). Soma-se a isso um gap salarial mediano entre homens e mulheres em Sênior (28,6%) e Especialista/Staff+ (22,2%) — mas não em Júnior/Pleno. Juntos, representatividade e remuneração são compatíveis com um desafio de diversidade que se manifesta de forma mais acentuada nos níveis de maior senioridade — um gradiente observado entre grupos diferentes de respondentes nesta amostra, e não uma relação causal ou uma trajetória acompanhada ao longo do tempo.

**Tecnologias predominantes.** SQL, Python e Java lideram as tecnologias mais citadas de 2025, com Power BI como ferramenta de BI mais adotada. Entre provedores de nuvem, a AWS assumiu a liderança a partir de 2024, superando a Azure, que liderava em 2023. A adoção de tecnologias mais avançadas (Databricks, Snowflake, Airflow, e nuvem de forma geral) cresce com a senioridade, com exceção parcial do Databricks entre os Especialistas/Staff+.

**Adoção de IA.** A parcela de empresas para quem IA Generativa não é prioridade caiu de 20,8% (2023) para 6,6% (2025), mas a parcela que a trata como principal frente do negócio também recuou no período — um sinal de que a adoção pode estar migrando para posições intermediárias de priorização, não capturadas pelo indicador binário disponível.

**Oportunidades e desafios.** Falta de expertise/recursos e dados não estarem prontos são os desafios que mais cresceram entre 2023 e 2025 para empresas que ainda não adotaram (ou adotam pouco) IA Generativa — reforçando que maturidade e governança de dados são pré-condições relevantes para projetos de IA. Já salário e flexibilidade/trabalho remoto se mantêm, de forma estável nos três anos, como os fatores mais citados para reter profissionais de Dados.